# Generate MiniLM Movie Embeddings (Title + Genres Only)

Since the TMDB 5000 dataset only covers a fraction of the 62,423 movies, we are dropping it entirely! 

Instead, this notebook dynamically constructs a text string for every single movie using its `title` and `genres` (e.g., `"Toy Story (1995). Adventure, Animation, Children, Comedy, Fantasy."`). It then feeds that rich string into the `all-MiniLM-L6-v2` neural network to generate a 384-dimensional dense semantic embedding.

**INSTRUCTIONS:**
1. Turn on the **GPU** accelerator in your Kaggle notebook settings.
2. Add the **MovieLens 25M Dataset** to the notebook.
3. Click "Run All"!

In [ ]:
!pip install -q sentence-transformers
import os
import numpy as np
import pandas as pd
import torch
from sentence_transformers import SentenceTransformer
from tqdm.auto import tqdm
import warnings
warnings.filterwarnings('ignore')

In [ ]:
print("Loading MovieLens Data...")

movies_path = '/kaggle/input/datasets/garymk/movielens-25m-dataset/ml-25m/movies.csv'
movies_df = pd.read_csv(movies_path)

# Replace the pipe separators in genres with commas for better NLP readability
movies_df['clean_genres'] = movies_df['genres'].str.replace('|', ', ')

# Handle the edge case where genres are missing or listed as "(no genres listed)"
movies_df['clean_genres'] = movies_df['clean_genres'].replace('(no genres listed)', '')

# Construct the dynamic semantic string
movies_df['semantic_text'] = movies_df['title'] + ". " + movies_df['clean_genres'] + "."

print(f"Total Movies to Encode: {len(movies_df)}")
print("\n--- Sample Semantic Strings ---")
for text in movies_df['semantic_text'].head(3):
    print(text)


In [ ]:
print("Initializing MiniLM model on GPU...")
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Using device: {device}")

model = SentenceTransformer('all-MiniLM-L6-v2', device=device)


In [ ]:
print("Generating Embeddings...")
texts_to_encode = movies_df['semantic_text'].tolist()

# Generate embeddings with a progress bar and batching
embeddings = model.encode(texts_to_encode, 
                          batch_size=256, 
                          show_progress_bar=True, 
                          convert_to_numpy=True)

print(f"Raw Embeddings Shape: {embeddings.shape}")

In [ ]:
print("Constructing final P_i.npy array...")
movie_ids = movies_df['movieId'].values.reshape(-1, 1)

# Concatenate movieId as the first column, and the 384 dimensions as the rest
P_i = np.hstack((movie_ids, embeddings))

print(f"Final P_i Shape: {P_i.shape}")

# Save to disk
np.save('P_i.npy', P_i)
print("Saved successfully to P_i.npy!")

In [ ]:
print("================================================")
print("         FINAL DIAGNOSTIC CHECK")
print("================================================")

emb_vectors = P_i[:, 1:]
variance = np.var(emb_vectors)

print(f"Variance of embeddings: {variance:.6f}")

if np.isnan(variance):
    print("🚨 FAILURE: Embeddings contain NaNs!")
elif variance == 0.0:
    print("🚨 FAILURE: Variance is 0.0! All embeddings are completely identical.")
else:
    print("✅ SUCCESS! Embeddings are mathematically valid and ready for TraitAlign.")
    
print("\nFirst 3 rows of embeddings (first 5 dimensions):")
print(emb_vectors[:3, :5])
